# Workspace
エージェント機能を持つチャットボットのサンプル。 mlflow と連携

In [1]:
import mlflow
import zoneinfo

tz_info = zoneinfo.ZoneInfo("Asia/Tokyo")
mlflow.set_experiment("agent-rag")

<Experiment: artifact_location='mlflow-artifacts:/5', creation_time=1767655484458, experiment_id='5', last_update_time=1767655484458, lifecycle_stage='active', name='agent-rag', tags={'mlflow.experimentKind': 'genai_development'}, workspace='default'>

# Playground

In [3]:
import datetime
import importlib
from mlflow.types.responses import Message, ResponsesAgentRequest

from agent_assistant import agent

importlib.reload(agent)

ymd = datetime.datetime.now(tz=tz_info).strftime("%Y%m%d_%H%M%S")
prompt = """
Databricks とその周辺の製品に関しての直近 (2025/02-) のニュースを振り返るため、まとめてください。
作業はタスクリストを作成し、上から順に step by step で処理していってください。

<instruction>
1. 対象となる期間を指定し検索。ニュースを取得
2. ニュースを製品毎に整理
3. 指定期間を通し特筆すべき動向を3つ以上抽出
4. ここまで調べ整理した内容を入力にレイアウトを整理した上で最終成果物を書出し
5. 最終成果物を読み、Output Requirement を満たすかを確認。満たせば処理完了。最終成果物を出力
</instruction>

<background>
現在日時: 2026/01/12

読み手は ITの専門的知識を持つ人間を想定。指定期間のIT業界の動向を振り返る必要が有る。
また、業界の動向のみならず同時期に発生したイベントやニュースも見る事で、広い視野で業界の動向を知る。
</background>

## Tool Guideline
- **search_news_history**: 月毎のニュースを取得するのに使用。対象月 (yyyy/MM) を指定し検索する

## Output Requirement
- 製品・作品単位でまとめてください
- 指定期間中の動向を複数の観点でまとめてください

例を以下に記載する。

```markdown
# 製品名A
- {製品Aのニュース1。100文字前後} ({yyyy/MM} ニュース日付を記載)
- {製品Aのニュース2。100文字前後} ({yyyy/MM} ニュース日付を記載)

# 製品名B
- {製品Bのニュース3。100文字前後} ({yyyy/MM} ニュース日付を記載)
- {製品Bのニュース4。100文字前後} ({yyyy/MM} ニュース日付を記載)

# IT業界外のニュース
- {他ニュース1。100文字前後} ({yyyy/MM} ニュース日付を記載)
- {他ニュース2。100文字前後} ({yyyy/MM} ニュース日付を記載)

# 期間中の動向 (対象製品周辺)
- 観点1
- 観点2

# 期間中の動向 (IT業界外)
- 観点1
- 観点2
```
"""

with mlflow.start_run(run_name=f"dev_{ymd}"):
    # 推論 (逐次)
    res = agent.agent_wrapped.predict(
        ResponsesAgentRequest(
            input=[
                Message(role="user", content=prompt),
            ]
        )
    )
    print(res.output)

[OutputItem(type='message', id='lc_run--019cc776-b90a-7f52-882c-8119368b49c8', content=[{'text': '承知いたしました。以下のタスクリストに沿って、Databricksおよびその周辺製品、さらには業界外の主要な動きについて、2025年2月から2026年1月現在までの情報を収集・整理し、報告書を作成します。\n\n### タスクリスト\n1.  **対象期間のニュース検索・取得**: `search_knowledge` を使用し、2025/02から2026/01までの月次ニュースを取得します。\n2.  **製品・カテゴリー別の整理**: 取得した情報を Databricks 関連製品、周辺IT製品、業界外ニュースに分類します。\n3.  **主要動向の抽出**: 指定期間を通した特筆すべき動向（Databricks周辺および業界外）を3つ以上抽出します。\n4.  **最終成果物の作成**: 指定されたレイアウトに従い、まとめを作成します。\n5.  **出力要件の確認**: 最終成果物が要件を満たしているか確認し、回答します。\n\nまずは、2025年2月からのニュースを順次検索します。\n\n#### Step 1: ニュースの取得 (2025/02 - 2026/01)\n\nまず、2025年2月から6月分を検索します。', 'type': 'output_text', 'annotations': []}], role='assistant'), OutputItem(type='function_call', id='6d7d1cd4-d47f-436e-b6c8-152dc1106c0e', call_id='6d7d1cd4-d47f-436e-b6c8-152dc1106c0e', name='search_knowledge', arguments='{"search_query": "2025/02"}'), OutputItem(type='function_call', id='2df03a8b-6c2e-4844-9ea5-8b7c9c5be011', call_id='2df03a8b-6c2e-4844-9ea5-8b7c9c5be011',

Trace(trace_id=tr-eed419ebdbdfb45671ab1929cfe8a84b)

In [4]:
print(len(res.output))
print(res.output[-1].content[0]["text"])

16
Databricks を中心とした 2025年2月から2026年1月現在までの主要なニュースと動向を整理しました。

# Databricks
- **Databricks Lakeflow の一般提供 (GA) 開始。** データエンジニアリングを統合し、インジェストから変換、オーケストレーションまでを一元管理する次世代パイプライン機能が正式にリリースされました。(2025/06)
- **Apache Iceberg の完全サポートと Lakebase 公開。** オープンなデータレイクハウスを推進するため、Iceberg との相互運用性を強化し、新たなメタデータ管理基盤 Lakebase のパブリックプレビューを開始。(2025/06)
- **Databricks Free Edition の提供開始。** 学習者や小規模開発者向けに、Unity Catalog を含むプラットフォーム機能を無料で体験できるエディションがリリースされました。(2025/06)
- **MLflow 3.0 への統合と Agent Evaluation 導入。** 生成AIエージェントの評価機能が MLflow に統合され、RAG やエージェントの品質測定を開発サイクルに組み込めるようになりました。(2025/11)
- **LSDP (Lakeflow Spark Declarative Pipelines) への移行。** 従来の Delta Live Tables (DLT) が Lakeflow の宣言型パイプラインとして統合され、より簡潔な定義が可能になりました。(2025/12)
- **Databricks Assistant へのエージェント機能追加。** ファイル操作やセルの自動生成を行う「Data Engineering Agent」などが導入され、開発体験が大幅に向上しました。(2026/01)

# LangChain / MLflow (周辺エコシステム)
- **LangChain v1.0 到達とエコシステムの刷新。** LangGraph との役割分担が明確化され、複雑な AI エージェント構築に向けた安定版として大規模なアップデートが行われました。(2025/12)
- **MLflow 3.0 リリース。** ChatModel から 

# Test

In [2]:
import datetime
import importlib
from agent_assistant import agent, evaluate

In [3]:
import datetime
import importlib
from agent_assistant import agent, evaluate

# テストデータを作成
eval_dataset = [
    {
        "inputs": {
            "messages": [
                {"role": "user", "content": "東京都の明日の天気を教えてください。"}
            ]
        },
        "expectations": {"expected_response": "東京都の明日の天気は晴れです"},
    },
    {
        "inputs": {
            "messages": [
                {"role": "user", "content": "横浜市の明日の天気を教えてください。"}
            ]
        },
        "expectations": {"expected_response": "横浜市の明日の天気は晴れです"},
    },
    {
        "inputs": {
            "messages": [
                {"role": "user", "content": "群馬県の明日の天気を教えてください。"}
            ]
        },
        "expectations": {"expected_response": "群馬県の天気は豪雨です。"},
    },
]

ymd = datetime.datetime.now(tz=tz_info).strftime("%Y%m%d_%H%M%S")
with mlflow.start_run(run_name=f"dev_{ymd}"):
    # モデルを評価
    importlib.reload(agent)
    res = evaluate.eval_responses(agent.agent_wrapped, eval_dataset)
    print(res)

2026/03/07 13:48:06 INFO mlflow.models.evaluation.agent_assistant.utils.trace: Auto tracing is temporarily enabled during the model evaluation for computing some metrics and debugging. To disable tracing, call `mlflow.autolog(disable=True)`.
2026/03/07 13:48:06 INFO mlflow.genai.agent_assistant.utils.data_validation: Testing model prediction with the first sample in the dataset. To disable this check, set the MLFLOW_GENAI_EVAL_SKIP_TRACE_VALIDATION environment variable to True.
2026/03/07 13:48:06 WARNING mlflow.tracing.fluent: Failed to start span predict_stream: 'NonRecordingSpan' object has no attribute 'context'. For full traceback, set logging level to debug.
2026/03/07 13:48:06 WARNING mlflow.tracing.fluent: Failed to start span LangGraph: 'NonRecordingSpan' object has no attribute 'context'. For full traceback, set logging level to debug.


Evaluating:   0%|          | 0/3 [Elapsed: 00:00, Remaining: ?] 

Invalid type dict in attribute 'token' value sequence. Expected one of ['bool', 'str', 'bytes', 'int', 'float'] or None
Invalid type dict in attribute 'token' value sequence. Expected one of ['bool', 'str', 'bytes', 'int', 'float'] or None
Invalid type dict in attribute 'token' value sequence. Expected one of ['bool', 'str', 'bytes', 'int', 'float'] or None
Invalid type dict in attribute 'token' value sequence. Expected one of ['bool', 'str', 'bytes', 'int', 'float'] or None
Invalid type dict in attribute 'token' value sequence. Expected one of ['bool', 'str', 'bytes', 'int', 'float'] or None
Invalid type dict in attribute 'token' value sequence. Expected one of ['bool', 'str', 'bytes', 'int', 'float'] or None


EvaluationResult(
  run_id: 5470d7bd7c9f4a2b835d8e74357200d8
  metrics:
    custom_check/mean: 1.0
    correctness/mean: 0.6666666666666666
  result_df: 3 rows x 15 cols
)


In [5]:
agent.agent_wrapped.predict({"input": [{"role": "user", "content": [{"type": "text", "text": "こんにちは！"}]}]})

ResponsesAgentResponse(tool_choice=None, truncation=None, id=None, created_at=None, error=None, incomplete_details=None, instructions=None, metadata=None, model=None, object='response', output=[OutputItem(type='message', id='lc_run--019cc88c-3984-73e2-8d0c-4f94faf85dba', content=[{'text': 'こんにちは！😊\n\nお疲れさまです。何かお手伝いできることはありますか？\n\n以下のようなことができます：\n- **天気情報の確認** - 特定の都市の天気をお調べします\n- **Obsidian vault の検索** - ノートの中から情報を検索します\n- **Obsidian ノートの取得** - 特定のノートの内容を表示します\n\n何かご質問やご依頼があれば、お気軽にお聞きください！', 'type': 'output_text', 'annotations': []}], role='assistant')], parallel_tool_calls=None, temperature=None, tools=None, top_p=None, max_output_tokens=None, previous_response_id=None, reasoning=None, status=None, text=None, usage=None, user=None, custom_outputs=None)

Trace(trace_id=tr-34573db173abce61727b7a17aba79e7e)